In [31]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import importlib
import som_class
importlib.reload(som_class)
from som_class import plot_u_matrix_hex, SelfOrganizingMapHex, plot_feature_map_hex, plot_weight_component_hex
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import shap
import os

In [26]:
data = pd.read_csv('Main Data_SOM.csv')
print(data.shape)
data.head()

(87, 532)


,Index,d__Archaea;__;__;__;__;__,d__Archaea;p__Euryarchaeota;c__Methanobacteria;o__Methanobacteriales;f__Methanobacteriaceae;g__Methanobrevibacter,d__Archaea;p__Euryarchaeota;c__Methanobacteria;o__Methanobacteriales;f__Methanobacteriaceae;g__Methanosphaera,d__Archaea;p__Halobacterota;c__Methanomicrobia;o__Methanomicrobiales;f__Methanocorpusculaceae;g__Methanocorpusculum,d__Archaea;p__Halobacterota;c__Methanomicrobia;o__Methanomicrobiales;f__Methanomicrobiaceae;g__Methanoculleus,d__Archaea;p__Halobacterota;c__Methanomicrobia;o__Methanomicrobiales;f__Methanomicrobiaceae;g__Methanomicrobium,d__Archaea;p__Halobacterota;c__Methanomicrobia;o__Methanomicrobiales;f__Methanomicrobiaceae;g__uncultured,d__Archaea;p__Halobacterota;c__Methanosarcinia;o__Methanosarciniales;f__Methanosarcinaceae;g__Methanimicrococcus,d__Archaea;p__Halobacterota;c__Methanosarcinia;o__Methanosarciniales;f__Methanosarcinaceae;g__Methanosarcina,...,d__Bacteria;p__Verrucomicrobiota;c__Verrucomicrobiae;o__Opitutales;f__Puniceicoccaceae;g__Verruc-01,d__Bacteria;p__Verrucomicrobiota;c__Verrucomicrobiae;o__Pedosphaerales;f__Pedosphaeraceae;g__DEV114,d__Bacteria;p__Verrucomicrobiota;c__Verrucomicrobiae;o__uncultured;f__uncultured;g__uncultured,d__Bacteria;p__Verrucomicrobiota;c__Verrucomicrobiae;o__Verrucomicrobiales;f__Akkermansiaceae;g__Akkermansia,d__Bacteria;p__Verrucomicrobiota;c__Verrucomicrobiae;o__Verrucomicrobiales;f__Rubritaleaceae;g__Luteolibacter,Unassigned;__;__;__;__;__,Acetic Acid,Butyric Acid,Hexanoic Acid,Methane
0,Zero (Phase 2),0,33,46,9,0,104,0,0,0,...,0,0,0,89,0,0,4404.172920,699.847659,12.449266,0.000
1,3R,0,30,7,24,0,0,0,0,0,...,0,0,0,24,0,0,5387.473707,963.449744,0.000000,7.989
2,5R,0,0,0,54,0,41,0,0,0,...,0,7,0,16,0,0,4155.655943,831.067581,15.527975,11.704
3,8R,0,0,0,182,0,28,0,0,0,...,0,10,0,13,0,0,5774.070236,1264.559493,8.366359,13.839
4,10R,0,0,0,385,0,0,0,25,0,...,0,18,0,14,2,0,7168.649163,1771.166592,20.882474,15.246


In [27]:
num_unknown = 0
num_uncultured = 0
renamed_columns = {}
name_counts = {}

for column in data.columns:
    if ";" in column:
        column_values = column.split(";")
        column_name = column_values[-1]
        if column_name == "__":
            num_unknown += 1
            column_name = f"unknown_{column_values[0]}_{num_unknown}"
        else:
            column_name = column_values[0] + "_" + column_values[-1]

        if column_name in name_counts:
            name_counts[column_name] += 1
            column_name = f"{column_name}_{name_counts[column_name]}"
        else:
            name_counts[column_name] = 1

        renamed_columns[column] = column_name

data.rename(columns=renamed_columns, inplace=True)
print(f"Renamed {len(renamed_columns)} columns")
print(data.shape)
data.head()

Renamed 527 columns
(87, 532)


,Index,unknown_d__Archaea_1,d__Archaea_g__Methanobrevibacter,d__Archaea_g__Methanosphaera,d__Archaea_g__Methanocorpusculum,d__Archaea_g__Methanoculleus,d__Archaea_g__Methanomicrobium,d__Archaea_g__uncultured,d__Archaea_g__Methanimicrococcus,d__Archaea_g__Methanosarcina,...,d__Bacteria_g__Verruc-01,d__Bacteria_g__DEV114,d__Bacteria_g__uncultured_41,d__Bacteria_g__Akkermansia,d__Bacteria_g__Luteolibacter,unknown_Unassigned_95,Acetic Acid,Butyric Acid,Hexanoic Acid,Methane
0,Zero (Phase 2),0,33,46,9,0,104,0,0,0,...,0,0,0,89,0,0,4404.172920,699.847659,12.449266,0.000
1,3R,0,30,7,24,0,0,0,0,0,...,0,0,0,24,0,0,5387.473707,963.449744,0.000000,7.989
2,5R,0,0,0,54,0,41,0,0,0,...,0,7,0,16,0,0,4155.655943,831.067581,15.527975,11.704
3,8R,0,0,0,182,0,28,0,0,0,...,0,10,0,13,0,0,5774.070236,1264.559493,8.366359,13.839
4,10R,0,0,0,385,0,0,0,25,0,...,0,18,0,14,2,0,7168.649163,1771.166592,20.882474,15.246


In [28]:
print(f"Data shape before drop: {data.shape}")
data = data.drop(columns="Index")
print(f"Data shape after drop: {data.shape}")
print(f"Data columns (first 10): {list(data.columns[:10])}")

target_columns = ["Acetic Acid", "Butyric Acid", "Hexanoic Acid", "Methane"]
data_columns = [col for col in data.columns if col not in target_columns]

print(f"Target columns found: {len(target_columns)}")
print(f"Data columns calculated: {len(data_columns)}")
print(f"Unique data columns: {len(set(data_columns))}")
print(f"Are there duplicates in data_columns? {len(data_columns) != len(set(data_columns))}")

if len(data_columns) != len(set(data_columns)):
    from collections import Counter
    col_counts = Counter(data_columns)
    duplicates = {col: count for col, count in col_counts.items() if count > 1}
    print(f"\nDuplicate columns found: {len(duplicates)}")
    for col, count in list(duplicates.items())[:10]:
        print(f"  '{col}': appears {count} times")

print(f"\nChecking if columns exist in data:")
missing_cols = [col for col in data_columns if col not in data.columns]
print(f"Missing columns: {len(missing_cols)}")
if missing_cols:
    print(f"Sample missing: {missing_cols[:5]}")

inputs = data[data_columns].copy()
targets = data[target_columns].copy()

print(f"\nFinal inputs shape: {inputs.shape}")
print(f"Final targets shape: {targets.shape}")
inputs

Data shape before drop: (87, 532)
Data shape after drop: (87, 531)
Data columns (first 10): ['unknown_d__Archaea_1', 'd__Archaea_g__Methanobrevibacter', 'd__Archaea_g__Methanosphaera', 'd__Archaea_g__Methanocorpusculum', 'd__Archaea_g__Methanoculleus', 'd__Archaea_g__Methanomicrobium', 'd__Archaea_g__uncultured', 'd__Archaea_g__Methanimicrococcus', 'd__Archaea_g__Methanosarcina', 'd__Archaea_g__Methanomassiliicoccus']
Target columns found: 4
Data columns calculated: 527
Unique data columns: 527
Are there duplicates in data_columns? False

Checking if columns exist in data:
Missing columns: 0

Final inputs shape: (87, 527)
Final targets shape: (87, 4)


,unknown_d__Archaea_1,d__Archaea_g__Methanobrevibacter,d__Archaea_g__Methanosphaera,d__Archaea_g__Methanocorpusculum,d__Archaea_g__Methanoculleus,d__Archaea_g__Methanomicrobium,d__Archaea_g__uncultured,d__Archaea_g__Methanimicrococcus,d__Archaea_g__Methanosarcina,d__Archaea_g__Methanomassiliicoccus,...,unknown_d__Bacteria_94,d__Bacteria_g__Cerasicoccus,d__Bacteria_g__Puniceicoccus,d__Bacteria_g__uncultured_40,d__Bacteria_g__Verruc-01,d__Bacteria_g__DEV114,d__Bacteria_g__uncultured_41,d__Bacteria_g__Akkermansia,d__Bacteria_g__Luteolibacter,unknown_Unassigned_95
0,0,33,46,9,0,104,0,0,0,0,...,0,0,0,0,0,0,0,89,0,0
1,0,30,7,24,0,0,0,0,0,0,...,0,0,0,0,0,0,0,24,0,0
2,0,0,0,54,0,41,0,0,0,0,...,33,0,0,71,0,7,0,16,0,0
3,0,0,0,182,0,28,0,0,0,0,...,61,0,0,146,0,10,0,13,0,0
4,0,0,0,385,0,0,0,25,0,0,...,144,0,0,318,0,18,0,14,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
83,0,8,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,40,0,0
84,0,5,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,34,0,0
85,0,0,0,3,0,0,0,0,0,0,...,0,0,0,0,0,0,0,14,0,0


In [29]:
def perform_shap_analysis(model, X_train, X_test, output_dir, title_prefix, n_top_features=10, n_force_samples=5):
    os.makedirs(output_dir, exist_ok=True)
    safe_prefix = title_prefix.replace(" ", "_")

    explainer = shap.TreeExplainer(model)
    shap_values_raw = explainer.shap_values(X_test)

    if isinstance(shap_values_raw, list):
        shap_values = np.array(shap_values_raw[0])
    else:
        shap_values = np.array(shap_values_raw)

    if hasattr(X_test, "columns"):
        feature_names = list(X_test.columns)
    else:
        feature_names = [f"f{i}" for i in range(shap_values.shape[1])]

    plt.figure()
    shap.summary_plot(
        shap_values,
        X_test,
        feature_names=feature_names,
        show=False
    )
    beeswarm_path = os.path.join(output_dir, f"{safe_prefix}_summary_beeswarm.png")
    plt.title(f"{title_prefix} SHAP Summary (Beeswarm)")
    plt.savefig(beeswarm_path, bbox_inches="tight", dpi=300)
    plt.close()

    plt.figure()
    shap.summary_plot(
        shap_values,
        X_test,
        feature_names=feature_names,
        plot_type="bar",
        show=False
    )
    bar_path = os.path.join(output_dir, f"{safe_prefix}_summary_bar.png")
    plt.title(f"{title_prefix} SHAP Summary (Bar)")
    plt.savefig(bar_path, bbox_inches="tight", dpi=300)
    plt.close()

    mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
    top_idx = np.argsort(-mean_abs_shap)[:n_top_features]

    for idx in top_idx:
        feat_name = feature_names[idx]
        plt.figure()
        shap.dependence_plot(
            ind=idx,
            shap_values=shap_values,
            features=X_test,
            feature_names=feature_names,
            show=False
        )
        dep_path = os.path.join(
            output_dir,
            f"{safe_prefix}_dependency_{feat_name.replace(' ', '_')}.png"
        )
        plt.title(f"{title_prefix} SHAP Dependence: {feat_name}")
        plt.savefig(dep_path, bbox_inches="tight", dpi=300)
        plt.close()

    n_samples = min(n_force_samples, shap_values.shape[0])

    base_value = explainer.expected_value
    if isinstance(base_value, (list, np.ndarray)):
        base_value = np.array(base_value).reshape(-1)[0]

    for i in range(n_samples):
        if hasattr(X_test, "iloc"):
            x_row = X_test.iloc[i, :]
        else:
            x_row = X_test[i, :]

        plt.figure()
        shap.plots._waterfall.waterfall_legacy(
            base_value,
            shap_values[i, :],
            feature_names=feature_names,
            max_display=min(len(feature_names), 20),
            show=False
        )
        waterfall_path = os.path.join(
            output_dir,
            f"{safe_prefix}_waterfall_sample_{i}.png"
        )
        plt.title(f"{title_prefix} SHAP Waterfall Plot - Sample {i}")
        plt.savefig(waterfall_path, bbox_inches="tight", dpi=300)
        plt.close()

    return shap_values


In [30]:
# fit a random forest model for each target measure feature importance
feature_importances = pd.DataFrame(index=target_columns, columns=data_columns)
normed_inputs = (inputs - inputs.min()) / (inputs.max() - inputs.min())
normed_outputs = (targets - targets.min()) / (targets.max() - targets.min())
X_train, X_test, y_train, y_test = train_test_split(normed_inputs, normed_outputs, test_size=0.2, random_state=42, shuffle=True)

for target in target_columns:
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train[target])
    importances = model.feature_importances_
    feature_importances.loc[target] = importances
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test[target], y_pred)
    r2 = r2_score(y_test[target], y_pred)

    # intentionally use train data for SHAP analysis so more samples are available
    shap_values = perform_shap_analysis(model, X_train, X_train, output_dir="shap_outputs", title_prefix=target, n_top_features=10, n_force_samples=5)
    print(f"{target} - MSE: {mse:.4f}, R2: {r2:.4f}")

feature_importances_mean = feature_importances.to_numpy().astype(float).mean(axis=0)
feature_importances_mean = feature_importances_mean / np.sum(feature_importances_mean)
feature_importances_ranked = pd.Series(feature_importances_mean, index=data_columns).sort_values(ascending=False)

print(feature_importances_ranked)

Acetic Acid - MSE: 0.0209, R2: 0.1198


/Users/benwilson/.pyenv/versions/aann/lib/python3.12/site-packages/shap/plots/_scatter.py:641: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig = plt.figure(figsize=figsize)
/var/folders/yz/s7yhnqx575nf_63f35422l8m0000gn/T/ipykernel_91429/2692113865.py:76: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure()
/var/folders/yz/s7yhnqx575nf_63f35422l8m0000gn/T/ipykernel_91429/2692113865.py:76: RuntimeWarning: More than 20 figures have been opened. Figures

Butyric Acid - MSE: 0.0322, R2: 0.3685


/var/folders/yz/s7yhnqx575nf_63f35422l8m0000gn/T/ipykernel_91429/2692113865.py:18: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure()
/var/folders/yz/s7yhnqx575nf_63f35422l8m0000gn/T/ipykernel_91429/2692113865.py:30: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure()
/var/folders/yz/s7yhnqx575nf_63f35422l8m0000gn/T/ipykernel_91429/2692113865.py:48: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot inter

Hexanoic Acid - MSE: 0.0068, R2: 0.7192
Methane - MSE: 0.0091, R2: 0.7749
d__Archaea_g__Methanosarcina                          0.114069
d__Bacteria_g__Petrimonas                             0.091988
d__Bacteria_g__Incertae_Sedis                         0.077529
d__Bacteria_g__Caproiciproducens                      0.033894
d__Bacteria_g__Peptostreptococcales-Tissierellales    0.031877
                                                        ...   
d__Bacteria_g__Alkalibacterium                        0.000000
d__Bacteria_g__Listeria                               0.000000
d__Bacteria_g__uncultured_19                          0.000000
d__Bacteria_g__Hungateiclostridiaceae                 0.000000
unknown_d__Bacteria_83                                0.000000
Length: 527, dtype: float64
Methane - MSE: 0.0091, R2: 0.7749
d__Archaea_g__Methanosarcina                          0.114069
d__Bacteria_g__Petrimonas                             0.091988
d__Bacteria_g__Incertae_Sedis                

...

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>